# Chapitre 14 · Compter ce qui coûte (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook
du chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans
le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le minimum repris de la leçon pour que les validations tournent en autonomie :
le GPT du chapitre 10 (pour la vérification contre `numel`) et les constantes du
run (contexte, débit crête du T4).

In [ ]:
import math

import torch
import torch.nn as nn

# La configuration exacte du chapitre 10.
vocab_size = 81      # 81 caractères distincts dans les fables
block_size = 64      # contexte : jusqu'à 64 caractères
d_model    = 96      # largeur du modèle
n_heads    = 4       # têtes d'attention
n_layers   = 2       # blocs Transformer empilés
d_ff       = 384     # dimension cachée du FFN (4 x d_model)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        split = lambda t: t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(x)), split(self.W_V(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float("-inf"))
        melange = torch.softmax(scores, dim=-1) @ V
        out = melange.transpose(1, 2).contiguous().view(B, T, d_model)
        return self.W_O(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))
        for bloc in self.blocs:
            h = bloc(h, self.masque)
        return self.tete(self.ln_final(h))


torch.manual_seed(42)
gpt = GPT()
T4_CRETE = 65e12   # ~65 TFLOPs/s crete (16 bits), chiffre constructeur
print("Mise en place terminee : GPT du chapitre 10 reconstruit.")

### Exercice 1 · La mémoire et les 6·N·D — niveau ●

Deux règles de poche, deux fonctions. `memoire_entrainement` : un paramètre à
entraîner réclame 4 nombres (poids, gradient, moyenne et variance d'AdamW), à
4 octets chacun en fp32, soit 16 octets par paramètre. `flop_count` : la règle
reine du compute, `C ≈ 6 · N · D`.

In [ ]:
def memoire_entrainement(n_params, octets_par_param=16):
    """Etat d'entrainement en octets : 16 octets/param en fp32."""
    return n_params * octets_par_param


def flop_count(n_params, n_tokens):
    """C ~ 6 * N * D, le compute total en FLOPs."""
    return 6 * n_params * n_tokens

In [ ]:
# Validation : mémoire et 6·N·D.
octets = memoire_entrainement(244_800)
assert octets is not ..., "remplace le ... par ton calcul"
assert octets == 3_916_800, f"244 800 x 16 = 3 916 800 octets, pas {octets}"
assert memoire_entrainement(1_000_000_000) == 16_000_000_000, "1 Md de params -> 16 Go"

n_steps, batch = 3000, 32
D = n_steps * batch * block_size                      # tokens vus pendant le run
C = flop_count(244_800, D)
assert C is not ..., "remplace le ... par ton calcul"
assert C == 9_024_307_200_000, f"C attendu 9.02e12, obtenu {C}"
print(f"Memoire : {octets/1e6:.2f} Mo | D = {D:,} tokens | C = {C:.2e} FLOPs".replace(",", " "))
print("Exercice 1 valide.")

### Exercice 2 · La calculatrice de temps — niveau ●●

Le temps réel = compute ÷ (débit crête × MFU). Complète `temps_estime` en trois
lignes : le compute avec `flop_count` (exercice 1), le débit réel en dégradant le
crête par le MFU, puis la division. La validation confronte l'erreur naïve
(MFU = 100 %) à l'estimation réaliste.

In [ ]:
def temps_estime(n_params, n_tokens, debit_crete_flops, mfu):
    """Temps d'entrainement estime, en secondes (mfu entre 0 et 1)."""
    compute = flop_count(n_params, n_tokens)
    debit_reel = debit_crete_flops * mfu
    return compute / debit_reel

In [ ]:
# Validation : la calculatrice de temps.
n_steps, batch = 3000, 32
D = n_steps * batch * block_size
t_naif = flop_count(244_800, D) / T4_CRETE   # MFU = 100 %, l'erreur classique
print(f"[naif, MFU=100%] {t_naif:.3f} s  (trop beau pour etre vrai)")

t30 = temps_estime(244_800, D, T4_CRETE, 0.3)
assert t30 is not ..., "remplace le ... par ton code"
assert abs(t30 - t_naif / 0.3) < 1e-6, "temps = compute / (crete * mfu)"
print(f"[T4, MFU=30%] {t30:.3f} s")
print("Exercice 2 valide.")

### Exercice 3 · Le verdict « ça rentre dans le T4 ? » — niveau ●●

Complète `rentre_dans_t4` (version compacte : elle renvoie le couple
`(ok, besoin_go)`). On compare l'état d'entraînement (`N × 16` octets, via
`memoire_entrainement` de l'exercice 1) au plafond du T4. Le cas de GPT-2 XL
doit **déborder**, et on le montre sans jamais rien allouer : un OOM calculé,
pas subi.

In [ ]:
T4_GO = 16.0


def rentre_dans_t4(n_params, plafond_go=T4_GO):
    """Verdict memoire, calcule AVANT de lancer. Renvoie (ok, besoin_go)."""
    besoin_go = memoire_entrainement(n_params) / 1e9
    ok = besoin_go <= plafond_go
    return ok, besoin_go

In [ ]:
# Validation : le verdict mémoire, calculé avant de lancer.
for nom, n in [("notre GPT", 244_800), ("GPT-2 small", 124_000_000),
               ("GPT-2 large", 774_000_000), ("GPT-2 XL", 1_500_000_000)]:
    res = rentre_dans_t4(n)
    assert res is not ..., "remplace le ... par ton code"
    ok, go = res
    print(f"{'OK ' if ok else '!! '}{nom:<14} {go:>6.2f} Go -> {'rentre' if ok else 'NE RENTRE PAS'}")

ok_gpt, _ = rentre_dans_t4(244_800)
ok_xl, go_xl = rentre_dans_t4(1_500_000_000)
assert ok_gpt is True, "notre GPT tient largement"
assert ok_xl is False and abs(go_xl - 24.0) < 0.1, "GPT-2 XL doit deborder (24 Go)"

# Le cas qui echoue, montre proprement (try/except, aucune allocation).
try:
    ok, go = rentre_dans_t4(1_500_000_000)
    if not ok:
        raise MemoryError(f"GPT-2 XL : {go:.1f} Go demandes pour 16 disponibles")
except MemoryError as e:
    print("\nRefuse AVANT de lancer :", e)
    print("-> aucun tenseur alloue. Trois multiplications, zero session gachee.")
print("Exercice 3 valide.")

### Exercice 4 · Compter les paramètres à la main — niveau ●●●

Le grand décompte, sans regarder la section 2. Écris `param_count`, qui renvoie
le **total** (un entier). Rappel : une matrice `(a, b)` a `a*b` paramètres, un
vecteur de taille `d` en a `d`. Les `nn.Linear` du FFN ont un biais ; l'attention
et la tête n'en ont pas ; chaque LayerNorm a deux vecteurs appris.

In [ ]:
def param_count(vocab_size, block_size, d_model, n_heads, n_layers, d_ff):
    """Compte les parametres d'un GPT decodeur, couche par couche."""
    tok_emb = vocab_size * d_model            # table des tokens (81 x 96)
    pos_emb = block_size * d_model            # table des positions (64 x 96)

    attn = 4 * (d_model * d_model)            # W_Q, W_K, W_V, W_O, sans biais
    ln = 2 * d_model                          # gamma + beta d'une LayerNorm
    ffn = (d_model * d_ff + d_ff) + (d_ff * d_model + d_model)  # W1+b1, W2+b2
    bloc = attn + ffn + 2 * ln                # deux LayerNorm par bloc

    ln_final = 2 * d_model                    # LayerNorm finale
    tete = d_model * vocab_size               # tete de sortie, sans biais

    total = tok_emb + pos_emb + n_layers * bloc + ln_final + tete
    return total

In [ ]:
# Validation : le décompte à la main contre numel.
a_la_main = param_count(vocab_size, block_size, d_model, n_heads, n_layers, d_ff)
mesure = sum(p.numel() for p in gpt.parameters())
assert a_la_main is not ..., "remplace le ... par ton calcul"
assert a_la_main == mesure == 244_800, (
    f"a la main = {a_la_main}, numel = {mesure} : les deux doivent valoir 244 800"
)
print("Exercice 4 valide : ton decompte a la main colle a numel. 244 800 parametres.")